# 01 - External and Standard Catalog Preflight

        Validate that the AIDP workspace can see the Oracle source through the external catalog and can create schemas in the target standard catalog.

        This notebook intentionally avoids source row counts so it does not scan the external catalog tables during preflight.

## Configuration

        Update the catalog and schema names if your AIDP workspace uses different names.

In [ ]:
# Edit these values for your AIDP workspace.
SOURCE_CATALOG = "aidp_sc_demo_source"
SOURCE_SCHEMA = "aidp_sc_demo"  # Use AIDP_SC_DEMO if your workspace exposes uppercase schema names.

TARGET_CATALOG = "aidp_sc_demo_standard"
SILVER_SCHEMA = "demo_supply_chain_silver"
GOLD_SCHEMA = "demo_supply_chain_gold"

# AIDP standard catalogs use managed Delta tables. Keep this as "delta" unless your tenancy requires a different table format.
TABLE_FORMAT = "delta"

In [ ]:
import re
from pyspark.sql import functions as F

IDENTIFIER_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def qident(value: str) -> str:
    """Quote and validate a catalog, schema, or table identifier."""
    if not IDENTIFIER_PATTERN.fullmatch(value):
        raise ValueError(f"Unsupported identifier: {value!r}")
    return f"`{value}`"


def qname(*parts: str) -> str:
    return ".".join(qident(part) for part in parts)


def show_small(df, n: int = 20) -> None:
    """Display a small result in notebook UI, falling back to show()."""
    try:
        display(df.limit(n))
    except NameError:
        df.show(n, truncate=False)


SOURCE_TABLES = {
    "suppliers": "aidp_sc_suppliers",
    "supplier_sites": "aidp_sc_supplier_sites",
    "item_categories": "aidp_sc_item_categories",
    "items": "aidp_sc_items",
    "po_headers": "aidp_sc_po_headers",
    "po_lines": "aidp_sc_po_lines",
    "blanket_prices": "aidp_sc_blanket_prices",
    "receipts": "aidp_sc_receipts",
    "invoice_lines": "aidp_sc_invoice_lines",
}


def source_table(key: str):
    return spark.table(qname(SOURCE_CATALOG, SOURCE_SCHEMA, SOURCE_TABLES[key]))


def target_table(schema: str, table: str) -> str:
    return qname(TARGET_CATALOG, schema, table)


def ensure_schema(schema: str) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qname(TARGET_CATALOG, schema)}")

## Spark runtime

In [ ]:
spark_version = spark.version
print(f"Spark version: {spark_version}")

major_minor = tuple(int(part) for part in spark_version.split(".")[:2])
if major_minor < (3, 5):
    raise RuntimeError("This demo expects Spark 3.5 or newer.")
if major_minor[0] > 4:
    print("Spark version is newer than the tested 3.5/4.x range; continue with standard Spark SQL APIs only.")

## Verify source tables

In [ ]:
source_namespace = qname(SOURCE_CATALOG, SOURCE_SCHEMA)
available_rows = spark.sql(f"SHOW TABLES IN {source_namespace}").collect()
available_tables = {row.tableName.lower() for row in available_rows}
required_tables = set(SOURCE_TABLES.values())
missing_tables = sorted(required_tables - available_tables)

print(f"Source namespace: {source_namespace}")
print(f"Required tables: {len(required_tables)}")
print(f"Visible tables: {len(available_tables)}")

if missing_tables:
    raise RuntimeError(f"Missing source tables in external catalog: {missing_tables}")

show_small(spark.createDataFrame([(name,) for name in sorted(required_tables)], ["required_source_table"]))

## Verify required columns without scanning rows

In [ ]:
required_columns = {
    "suppliers": {"SUPPLIER_ID", "SUPPLIER_NUMBER", "SUPPLIER_NAME", "STATUS", "DEFAULT_CURRENCY"},
    "supplier_sites": {"SUPPLIER_SITE_ID", "SUPPLIER_ID", "SITE_CODE", "COUNTRY_CODE", "CURRENCY_CODE"},
    "item_categories": {"CATEGORY_ID", "CATEGORY_CODE", "CATEGORY_NAME"},
    "items": {"ITEM_ID", "ITEM_NUMBER", "ITEM_DESCRIPTION", "CATEGORY_ID", "PRIMARY_UOM", "BASE_UNIT_PRICE"},
    "po_headers": {"PO_HEADER_ID", "PO_NUMBER", "SUPPLIER_ID", "SUPPLIER_SITE_ID", "BUYER_NAME", "CURRENCY_CODE", "ORDER_DATE", "PO_STATUS"},
    "po_lines": {"PO_LINE_ID", "PO_HEADER_ID", "LINE_NUMBER", "ITEM_ID", "ORDERED_QUANTITY", "UOM_CODE", "UNIT_PRICE", "NEED_BY_DATE"},
    "blanket_prices": {"BLANKET_PRICE_ID", "SUPPLIER_ID", "SUPPLIER_SITE_ID", "ITEM_ID", "UOM_CODE", "CURRENCY_CODE", "UNIT_PRICE", "EFFECTIVE_START_DATE", "EFFECTIVE_END_DATE"},
    "receipts": {"RECEIPT_ID", "PO_LINE_ID", "RECEIPT_NUMBER", "RECEIPT_DATE", "RECEIVED_QUANTITY", "REJECTED_QUANTITY", "RETURNED_QUANTITY", "RECEIPT_STATUS"},
    "invoice_lines": {"INVOICE_LINE_ID", "PO_LINE_ID", "INVOICE_NUMBER", "INVOICE_DATE", "INVOICED_QUANTITY", "INVOICE_UNIT_PRICE", "MATCH_STATUS"},
}

column_issues = []
for key, expected in required_columns.items():
    actual = {field.name.upper() for field in source_table(key).schema.fields}
    missing = sorted(expected - actual)
    if missing:
        column_issues.append((SOURCE_TABLES[key], ", ".join(missing)))

if column_issues:
    show_small(spark.createDataFrame(column_issues, ["table_name", "missing_columns"]))
    raise RuntimeError("One or more source tables are missing required columns.")

print("All required source columns are visible.")

## Prepare target schemas

In [ ]:
ensure_schema(SILVER_SCHEMA)
ensure_schema(GOLD_SCHEMA)
spark.sql(f"SHOW SCHEMAS IN {qident(TARGET_CATALOG)}").show(truncate=False)

## Result

        Preflight is complete when the required source tables are visible and the Silver and Gold schemas exist in the standard catalog.